# 04 Scans, optimisation, matching

Each code cell below is one of the five scripts of `04_scans_optimisation/`, in order: the same code as the command line (`python 04_scans_optimisation/NN_name.py`), plus a first line setting `__file__`, which the scripts use to find `output/`. Change a cell to experiment; to change an example for good, edit its script and run `python sync_notebooks.py`, which copies the scripts into the notebooks. Each script's docstring says what it shows, what it needs and what it checks.

20 and 21 use a one-quad lattice built in Python (with `02_tracking/simple_lattice.py`) and check against matrix optics. 22–24 run JFEL from laura-lattices: the bunch compressor (22, 24) and the match into the undulator (23), checked against the JFEL README.

Everything here is tier A (pip only); 22 adds an elegant check when docker and the simcodes image are there. 22 takes about 80 s, the others under 20 s.

## 20 Measure the emittance with a quad scan

Vary one quad's `k1l` with `modifyElement`, track, and record $\sigma^2$ on a screen a distance $L$ downstream. With the quad as a thin lens of strength $k$, $\sigma^2 = A k^2 + B k + C$, and the parabola's coefficients give the beam's sigma matrix at the quad, so its emittance $\sqrt{\Sigma_{11}\Sigma_{22} - \Sigma_{12}^2}$. The exact fit uses the thick-quad matrices instead: $\sigma^2$ is linear in $\Sigma_{11}, \Sigma_{12}, \Sigma_{22}$.

The exact fit gives back the beam's emittance to $5\times10^{-6}$ in both planes. The thin-lens fit is 1% off in $x$, where the scan passes through the waist, but 43% off in $y$, where $\sigma_y$ only grows: there the fit is ill-conditioned and the small model error is amplified. A real scan should span the waist.

In [ ]:
__file__ = "20_quad_scan_emittance.py"  # python and %run set this; a notebook cell does not
"""
20 - Measure the emittance with a quadrupole scan
=================================================

Shows: a quad scan, the standard way to measure emittance. Vary one
quadrupole's strength (modifyElement), record the beam size on a screen
downstream after each track(), and fit. The fit gives the beam's sigma
matrix at the quad, and so its emittance (and Twiss parameters).

Run:    python 04_scans_optimisation/20_quad_scan_emittance.py
Tier:   A
Needs:  SIMBA, LAURA and Ocelot.
Output: output/20_quad_scan_emittance/: the lattice, run/ (the last scan
        point) and quad_scan.png.

Units: k1l in 1/m; sizes in m; emittance (geometric) in m rad.

The line: a 0.1 m quadrupole Q centred 0.5 m from the start, then a screen
L = 3 m beyond its centre. The beam: 100 MeV, beta = 10 m and alpha = 0 at
the start. One scan gives both planes: Q focuses in x (sigma_x passes
through a minimum) and defocuses in y.

Two fits:
- The textbook one treats Q as a thin lens, M = [[1 - k L, L], [-k, 1]]
  with k = k1l. Then sigma^2 = A k^2 + B k + C, a parabola, and the sigma
  matrix at the quad is S11 = A / L^2, S12 = -(B + 2 L S11) / (2 L^2),
  S22 = (C - S11 - 2 L S12) / L^2, with emittance sqrt(S11 S22 - S12^2).
- The exact one uses the thick-quad matrices from the start to the screen
  (02_tracking/simple_lattice.transfer_matrix). sigma^2 is linear in the
  three elements of the starting sigma matrix, so a linear least-squares
  fit gives them.

Check: the exact fit's emittance must equal the starting beam's own
(sqrt(det Sigma0), from its particles) to 1e-3 in both planes. Tracking has
no noise here, and the second-order chromatic terms the matrices leave out
are ~1e-5 (measured 5e-6). The thin-lens fit must be within 5% in x, where
the scan passes through the waist: its error comes from the quad's length
(measured -1.1%). In y it is only printed: sigma_y grows over the whole
scan, the parabola's minimum is far outside it, and the fit is
ill-conditioned. The same ~1% model error then becomes +43% (fed synthetic
thin-lens data instead, both planes come back exact). The lesson for a real
scan: span the waist. A real measurement also adds the screen's noise and
resolution, which this leaves out.
"""

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
from simple_lattice import make_beam, marker, quad, sigma_matrix, transfer_matrix, write_master_lattice  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
Z_QUAD, L, QUAD_LENGTH = 0.5, 3.0, 0.1
Z_SCREEN = Z_QUAD + L

files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "SCREEN"},
    }
}
elements = [marker("START", 0.0), quad("Q", Z_QUAD, 0.0, QUAD_LENGTH), marker("SCREEN", Z_SCREEN)]
settings = write_master_lattice(LATTICE, elements, files, "scan.def")
start = make_beam(LATTICE / "START.openpmd.hdf5", beta=(10.0, 10.0), alpha=(0.0, 0.0))

framework = fw.Framework(master_lattice=str(LATTICE), directory=str(OUT / "run"), clean=True, verbose=False)
framework.loadSettings(settings)

strengths = np.linspace(0.1, 0.6, 11)
size2 = {"x": [], "y": []}
for k1l in strengths:
    framework.modifyElement("Q", "k1l", float(k1l))
    framework.track()
    screen = rbf.load_file(str(OUT / "run" / "SCREEN.openpmd.hdf5"))
    for plane in ("x", "y"):
        size2[plane].append(np.var(getattr(screen, plane).val))


def thin_lens_fit(k, s2, sign):
    """Emittance from the parabola sigma^2(k); sign -1 for y, where the quad's k is -k1l."""
    a, b, c = np.polyfit(sign * k, s2, 2)
    s11 = a / L ** 2
    s12 = -(b + 2 * L * s11) / (2 * L ** 2)
    s22 = (c - s11 - 2 * L * s12) / L ** 2
    return np.sqrt(s11 * s22 - s12 ** 2)


def exact_fit(k, s2, plane):
    """Sigma matrix at the start from the thick-quad matrices; returns its emittance."""
    rows = []
    for k1l in k:
        m = transfer_matrix([quad("Q", Z_QUAD, k1l, QUAD_LENGTH)], plane, 0.0, Z_SCREEN)
        rows.append([m[0, 0] ** 2, 2 * m[0, 0] * m[0, 1], m[0, 1] ** 2])
    s11, s12, s22 = np.linalg.lstsq(np.array(rows), s2, rcond=None)[0]
    return np.sqrt(s11 * s22 - s12 ** 2)


print(f"{'plane':5s} {'beam eps [nm]':>13s} {'exact fit':>10s} {'rel diff':>9s} {'thin lens':>10s} {'rel diff':>9s}")
problems = []
for plane, sign in (("x", 1), ("y", -1)):
    s2 = np.array(size2[plane])
    eps = np.sqrt(np.linalg.det(sigma_matrix(start, plane)))
    exact, thin = exact_fit(strengths, s2, plane), thin_lens_fit(strengths, s2, sign)
    print(f"{plane:5s} {eps * 1e9:13.4f} {exact * 1e9:10.4f} {exact / eps - 1:9.1e} {thin * 1e9:10.4f} {thin / eps - 1:9.1e}")
    if abs(exact / eps - 1) > 1e-3:
        problems.append(f"{plane}: exact fit {exact:.4e}, beam {eps:.4e}")
    if plane == "x" and abs(thin / eps - 1) > 0.05:
        problems.append(f"{plane}: thin-lens fit {thin:.4e}, beam {eps:.4e}")
print("(y's thin-lens fit is ill-conditioned: the scan does not pass through a y waist)")

for plane in ("x", "y"):
    plt.plot(strengths, np.sqrt(size2[plane]) * 1e6, "o-", label=f"sigma_{plane}")
plt.xlabel("Q k1l [1/m]")
plt.ylabel("beam size at the screen [um]")
plt.legend()
plt.savefig(OUT / "quad_scan.png", dpi=120, bbox_inches="tight")
print(f"\nPlot: {OUT / 'quad_scan.png'}")

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print("\nOK: the quad scan gives back the beam's emittance in both planes")

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/20_quad_scan_emittance/quad_scan.png"))

## 21 Optimise a setting with scipy

Wrap "set, track, read the screen" in a function and give it to `scipy.optimize.minimize_scalar`. It finds the quad strength for the smallest $\sigma_x$ on the screen in 10 tracks, the same as the thick-quad matrix model to $3\times10^{-6}$; the thin-lens answer $k = 1/L + \Sigma_{12}/\Sigma_{11}$ is 0.6% off. Every trial goes to a CSV, and `save_changes_file` keeps the best setting for `load_changes_file` later.

In [ ]:
__file__ = "21_optimise_scipy.py"  # python and %run set this; a notebook cell does not
"""
21 - Optimise a setting with scipy
==================================

Shows: wrapping track() in a function and giving it to an optimiser. The
optimiser (scipy.optimize.minimize_scalar) chooses quadrupole Q's k1l to
make the beam smallest in x on a screen downstream. Each call sets k1l
(modifyElement), tracks and reads the screen's beam back. Every trial is
logged to a CSV, and the best setting is saved with save_changes_file, so
it can be applied later with load_changes_file (as in 15).

Run:    python 04_scans_optimisation/21_optimise_scipy.py
Tier:   A
Needs:  SIMBA, LAURA and Ocelot.
Output: output/21_optimise_scipy/: the lattice, run/, trials.csv and
        best.yaml (the changes file).

Units: k1l in 1/m; sizes in m.

The line is 20's: a 0.1 m quad 0.5 m from the start, and a screen L = 3 m
beyond its centre, with a 100 MeV beam, beta = 10 m and alpha = 0 at the
start. For a thin lens the smallest sigma_x is at k = 1/L + S12/S11, with
S the sigma matrix at the quad (k = 1/L - alpha/beta there).

Check: the optimiser's k1l must equal the optimum of the thick-quad matrix
model (02_tracking/simple_lattice, applied to the starting beam's sigma
matrix) to 1e-4 relative. The two differ only by the second-order
chromatic terms the matrices leave out, and by the optimiser's tolerance
(xatol 1e-6); measured 3e-6, after 10 tracks. The thin-lens value is
printed: it is off by 0.6% (the quad's length). The changes file must hold
the best k1l.
"""

import csv
import sys
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
import yaml
from scipy.optimize import minimize_scalar

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
from simple_lattice import make_beam, marker, quad, sigma_matrix, transfer_matrix, write_master_lattice  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
Z_QUAD, L, QUAD_LENGTH = 0.5, 3.0, 0.1
Z_SCREEN = Z_QUAD + L
BOUNDS = (0.0, 1.0)

files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "SCREEN"},
    }
}
elements = [marker("START", 0.0), quad("Q", Z_QUAD, 0.0, QUAD_LENGTH), marker("SCREEN", Z_SCREEN)]
settings = write_master_lattice(LATTICE, elements, files, "optimise.def")
start = make_beam(LATTICE / "START.openpmd.hdf5", beta=(10.0, 10.0), alpha=(0.0, 0.0))

framework = fw.Framework(master_lattice=str(LATTICE), directory=str(OUT / "run"), clean=True, verbose=False)
framework.loadSettings(settings)
trials = []


def sigma_x_at_screen(k1l):
    """The objective: set Q, track, return sigma_x on the screen."""
    framework.modifyElement("Q", "k1l", float(k1l))
    framework.track()
    size = float(np.std(rbf.load_file(str(OUT / "run" / "SCREEN.openpmd.hdf5")).x.val))
    trials.append((float(k1l), size))
    return size


result = minimize_scalar(sigma_x_at_screen, bounds=BOUNDS, method="bounded", options={"xatol": 1e-6})
best = float(result.x)
with open(OUT / "trials.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["k1l", "sigma_x"])
    writer.writerows(trials)

# The same optimum from the matrix model
s0 = sigma_matrix(start, "x")


def model_size2(k1l):
    m = transfer_matrix([quad("Q", Z_QUAD, k1l, QUAD_LENGTH)], "x", 0.0, Z_SCREEN)
    return (m @ s0 @ m.T)[0, 0]


model = float(minimize_scalar(model_size2, bounds=BOUNDS, method="bounded", options={"xatol": 1e-9}).x)
drift = np.array([[1.0, Z_QUAD], [0.0, 1.0]])
s_quad = drift @ s0 @ drift.T
thin = 1 / L + s_quad[0, 1] / s_quad[0, 0]

# Leave the best setting in place, then save it
framework.modifyElement("Q", "k1l", best)
changes_file = OUT / "best.yaml"
framework.save_changes_file(filename=str(changes_file))
saved = yaml.safe_load(changes_file.read_text())["Q"]

print(f"optimiser:  k1l = {best:.6f} 1/m after {len(trials)} tracks, sigma_x = {result.fun * 1e6:.3f} um")
print(f"matrices:   k1l = {model:.6f} 1/m   (relative difference {best / model - 1:.1e})")
print(f"thin lens:  k1l = {thin:.6f} 1/m   (relative difference {thin / model - 1:.1e})")
print(f"\n{changes_file.name}:\n" + yaml.safe_dump({"Q": saved}))

# --- Check -----------------------------------------------------------------
assert result.success, result.message
assert abs(best / model - 1) < 1e-4, f"optimiser {best}, matrix model {model}"
saved_k1l = saved["magnetic"]["multipoles"]["K1L"]["normal"]["new"]
assert np.isclose(saved_k1l, best, rtol=1e-12), f"changes file has {saved_k1l}, best is {best}"
print("OK: the optimiser finds the matrix model's optimum, and the changes file holds it")

## 22 Optimise the JFEL compressor for 1000 A

`scipy.optimize.brentq` adjusts the L4H harmonic cavity's phase until the peak current after JFEL's chicane is 1000 A (Ocelot, structure wakes on): about $-180.84°$, in 7 tracks. The optimum sits on the edge of full compression, where $\pm 0.2°$ moves the current by 8%.

Then the JFEL README's warning, shown: the model must include what shapes the compression. Without the structure wakes, Ocelot and Cheetah agree with each other (2%), and both give half the current. An optimisation in a code without wakes would land somewhere else. elegant (if docker is there) shows the same wake effect.

In [ ]:
__file__ = "22_optimise_compression.py"  # python and %run set this; a notebook cell does not
"""
22 - Optimise the JFEL bunch compressor for a target peak current
=================================================================

Shows: optimising a real lattice. JFEL's L4H harmonic cavity sets the
energy chirp the chicane compresses. A root finder (scipy brentq) adjusts
its phase until the peak current after the chicane is 1000 A. It then
re-runs the answer in other codes, which shows the JFEL README's warning
that the model must include what shapes the compression: here, the
structure wakes.

Run:    python 04_scans_optimisation/22_optimise_compression.py
Tier:   A (Ocelot, Cheetah); B for the elegant check (docker).
Needs:  SIMBA, LAURA, laura-lattices (JFEL), Ocelot and Cheetah; docker and
        ghcr.io/astec-stfc/simcodes-docker:master for elegant (skipped
        otherwise).
Output: output/22_optimise_compression/: one directory per code.

Units: phase in degrees (SIMBA's convention: -180 is the decelerating
crest); peak current in A; sigma_t in fs.

The line is JFEL's Lattices/jfel_compression.def ("Compressor", from the
L4H cavity through the chicane), at the nominal chicane angle (0.13 rad)
and amplitude, with CSR off (it doubles the run time and changes little
here) and the structure wakes on. The figures of merit are as in JFEL's
Scripts/compression.py: the charge-weighted sigma_t, and the peak of a
200-bin current histogram smoothed over 3 bins. "Spikiness" is the peak
over a flat-top bunch's current, Q / (sqrt(12) sigma_t).

At the nominal phase (-184) the bunch is not compressed enough (~480 A);
at -180 it is past the edge of full compression (~1560 A). brentq brackets
[-184, -180] and stops at 0.01 deg.

Checks:
- Ocelot (wakes on) reaches 1000 A within 1%. The 0.01 deg tolerance is
  ~4 A at the slope here (~400 A/deg); measured 1001 A, in 7 tracks.
- At the same phase, Cheetah (which has no structure wakes) agrees with
  Ocelot run without wakes to 5% (measured 2%): with the same physics the
  codes agree.
- The wakes change the answer: Ocelot's peak current is >1.5x higher with
  them than without (measured 2.05x). An optimisation in a code without
  wakes (Cheetah, MAD-X) would land somewhere else entirely.
- elegant (docker) shows the same: >1.5x with the wakes (measured 2.8x).
  Its current with the wakes is printed, not checked: measured 1547 A
  against Ocelot's 999 A. The README quotes ~40% for the same comparison
  with CSR on. The optimum sits on the edge of full compression, where
  +-0.2 deg moves the current by ~8% (printed below), so small differences
  between the codes' RF and wake models show up large.

The README's optimisation also varies the chicane angle and the amplitude
(optimise_compression.py --target 1000, 1010 A). One target needs one knob,
so this keeps the other two at nominal. The README's sigma_t values were
made with an older SIMBA, and today's runs differ (README "~190 fs, 450 A"
at nominal, measured 182 fs, 480 A); the checks above don't use them.
"""

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import JFEL
from scipy.optimize import brentq

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
JFEL_DIR = os.path.dirname(JFEL.__file__)
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"
L4H, END = "JFEL-L4H-LIN-CAV-01", "JFEL-S06-SIM-MARK-01"
TARGET = 1000.0


def docker_ready():
    return bool(shutil.which("docker")) and subprocess.run(
        ["docker", "image", "inspect", IMAGE], capture_output=True).returncode == 0


def compressor(code, tag, wakes=True, **kwargs):
    """JFEL's compression line in `code`, with CSR off."""
    f = fw.Framework(master_lattice=JFEL_DIR, directory=str(OUT / tag), clean=True, verbose=False, **kwargs)
    f.loadSettings("Lattices/jfel_compression.def")
    f.change_Lattice_Code("Compressor", code)
    f["Compressor"].csr_enable = False
    f["Compressor"].wakefield_enable = wakes
    return f


def compress(f, tag, phase):
    """Track at an L4H phase; return the peak current [A], sigma_t [fs] and spikiness."""
    f.modifyElement(L4H, "phase", float(phase))
    f.track(files=["Compressor"], check_lattice=False)
    beam = rbf.load_file(str(OUT / tag / f"{END}.openpmd.hdf5"))
    q, t = np.abs(beam.Particles.charge.val), beam.t.val
    sigma_t = np.sqrt(np.cov(t, aweights=q))
    counts, edges = np.histogram(t, bins=200, weights=q)
    peak = np.convolve(counts / (edges[1] - edges[0]), np.ones(3) / 3, mode="same").max()
    spiky = peak / (q.sum() / (np.sqrt(12) * sigma_t))
    print(f"  {tag:16s} phase {phase:9.3f}: I = {peak:6.0f} A, sigma_t = {sigma_t * 1e15:5.1f} fs, spikiness {spiky:.2f}")
    return peak


print("Ocelot, wakes on: solving for the L4H phase")
ocelot = compressor("ocelot", "ocelot")
phase = brentq(lambda p: compress(ocelot, "ocelot", p) - TARGET, -184.0, -180.0, xtol=0.01)
current = compress(ocelot, "ocelot", phase)
print(f"solution: L4H phase {phase:.3f} deg\n\nThe edge of compression, +-0.2 deg:")
for p in (phase - 0.2, phase + 0.2):
    compress(ocelot, "ocelot", p)

print("\nThe same phase without structure wakes:")
no_wakes = compress(compressor("ocelot", "ocelot_no_wakes", wakes=False), "ocelot_no_wakes", phase)
cheetah = compress(compressor("cheetah", "cheetah"), "cheetah", phase)

elegant = None
if docker_ready():
    print("\nelegant (docker):")
    elegant = {w: compress(compressor("elegant", f"elegant_{t}", wakes=w, container_runtime="docker"), f"elegant_{t}", phase)
               for w, t in ((True, "wakes"), (False, "no_wakes"))}
    print(f"elegant with wakes / Ocelot with wakes = {elegant[True] / current:.2f} (not checked; see the docstring)")
else:
    print(f"\nelegant check skipped: docker or {IMAGE} not available")

# --- Checks ----------------------------------------------------------------
print()
assert abs(current / TARGET - 1) < 0.01, f"Ocelot reached {current:.0f} A, target {TARGET:.0f} A"
assert abs(cheetah / no_wakes - 1) < 0.05, f"Cheetah {cheetah:.0f} A, Ocelot without wakes {no_wakes:.0f} A"
assert current / no_wakes > 1.5, f"wakes changed Ocelot's current only {current / no_wakes:.2f}x"
if elegant:
    assert elegant[True] / elegant[False] > 1.5, f"wakes changed elegant's current only {elegant[True] / elegant[False]:.2f}x"
print(f"OK: {current:.0f} A at L4H {phase:.2f} deg; without wakes the codes agree "
      f"(Cheetah/Ocelot {cheetah / no_wakes:.3f}), and the wakes multiply the current by {current / no_wakes:.2f}")

## 23 Match into the undulator with a `match:` block

JFEL's `jfel_match_undulator.def` has a `match:` block on its S13 line: four quads as variables, targets at the undulator entrance. Before tracking that line, SIMBA runs Ocelot's matcher and writes the quad strengths back. The target is the undulator's periodic Twiss, from a SIMBA line with `tracking: {periodic: True}` and `ring_parameters()`.

From half strength the matcher returns to the strengths stored in the lattice, and the beam reproduces the README's table. CSR is turned off, as in the README's match: on main the `.def` would include it (`ISSUES.md` #28).

In [ ]:
__file__ = "23_match_with_def.py"  # python and %run set this; a notebook cell does not
"""
23 - Match into the JFEL undulator with a `match:` block
========================================================

Shows: SIMBA's matcher, set up in the .def file. JFEL's
Lattices/jfel_match_undulator.def has three lines: Linac (the injector
beam to S13), Match (S13 to the undulator entrance) and Undulator. The
Match line has a block

    match:
      enable: True
      variables: [JFEL-S13-MAG-QUAD-01, ..., -04]
      targets: {}

When it loads that line's input beam, SIMBA's matching (simba.Modules.Matching)
varies the listed quads to hit the targets, from that beam's Twiss, and writes
their k1l into the lattice; then the line is written and tracked. The match
runs in the line's own code, here Ocelot, with Ocelot's matcher; `code:` in
the block picks another, and 06_matching shows the rest (every code, matrix
targets, chicanes, the RF). Targets are SIMBA's Twiss names. They are filled
in from code here, as JFEL's match_to_undulator.py does:

    framework["Match"].file_block["match"]["targets"] = {ELEMENT: {"beta_x": ..., ...}}

The target is the undulator's periodic Twiss, also from SIMBA: a line with
`tracking: {periodic: True}` over the undulator, written to a small .def
next to the run, then `framework["Periodic"].ring_parameters()`.

Run:    python 04_scans_optimisation/23_match_with_def.py
Tier:   A
Needs:  SIMBA, LAURA, laura-lattices (JFEL) and Ocelot.
Output: output/23_match_with_def/: run/ (the three lines) and periodic/.

Units: beta in m; k1l in 1/m.

CSR is off on every line, and the structure wakes stay on, as in the
README's match. The S13 strengths stored in the lattice were matched
without CSR, and since SIMBA PR #67 the .def as it stands includes it
(ISSUES.md #28). The wakes matter through the energy: without them the
beam reaches the undulator ~0.7% higher in energy, its natural focusing is
weaker, and the periodic beta_x is 0.7% larger. The quads start at half
their stored strengths, so the matcher has real work to do.

Checks:
- The tracked beam at the undulator entrance and exit equals the periodic
  target to 2% in beta and 0.02 in alpha (measured 0.4%, 0.009). They
  aren't equal: the matcher works from the beam's Twiss at S13 and linear
  optics, while the tracked beam has energy spread and the S13 quads and
  undulator are chromatic.
- The periodic Twiss and the beam at the entrance and exit equal the JFEL
  README's "Matching into the undulator" table to 1e-3 in beta and 0.002
  in alpha. The README's values are this calculation rounded to 3
  decimals (measured within 1.5e-4 and 0.0005).
- The matched k1l equal the strengths stored in the lattice, which came
  from the same match, to 1e-3 (measured 7e-6).
"""

import os
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
import yaml
from laura_lattices import JFEL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
RUN = OUT / "run"
JFEL_DIR = os.path.dirname(JFEL.__file__)
QUADS = [f"JFEL-S13-MAG-QUAD-0{i}" for i in range(1, 5)]
S13, ENTRANCE, EXIT = "JFEL-S13-SIM-MARK-01", "JFEL-FEL-SIM-MARK-01", "JFEL-FEL-SIM-MARK-02"
KEYS = ("beta_x", "alpha_x", "beta_y", "alpha_y")
README = {  # JFEL README, "Matching into the undulator"
    "periodic": dict(zip(KEYS, (3.448, 0.185, 2.208, -0.274))),
    "entrance": dict(zip(KEYS, (3.460, 0.184, 2.211, -0.280))),
    "exit": dict(zip(KEYS, (3.461, 0.181, 2.204, -0.283))),
}


def twiss(name):
    t = rbf.load_file(str(RUN / f"{name}.openpmd.hdf5")).twiss
    return {k: float(np.ravel(getattr(t, k).val)[0]) for k in KEYS}


def differences(got, want):
    """Largest relative beta difference and absolute alpha difference."""
    beta = max(abs(got[k] / want[k] - 1) for k in ("beta_x", "beta_y"))
    alpha = max(abs(got[k] - want[k]) for k in ("alpha_x", "alpha_y"))
    return beta, alpha


framework = fw.Framework(master_lattice=JFEL_DIR, directory=str(RUN), clean=True, verbose=False)
framework.loadSettings("Lattices/jfel_match_undulator.def")
for line in ("Linac", "Match", "Undulator"):
    framework[line].csr_enable = False
stored = [framework.getElement(q).k1l for q in QUADS]
framework.track(files=["Linac"], check_lattice=False)

# The undulator's periodic Twiss. The input beam (for its energy) is the
# Linac's output, given as an absolute path without its extension.
periodic_def = OUT / "periodic.def"
periodic_def.write_text(yaml.safe_dump({
    "files": {"Periodic": {
        "code": "ocelot",
        "input": {"particle_definition": str(RUN / S13)},
        "tracking": {"periodic": True},
        "output": {"start_element": ENTRANCE, "end_element": EXIT},
    }},
    "layout": "./layouts.yaml", "section": "./sections.yaml", "element_list": "./YAML/",
}, sort_keys=False))
periodic = fw.Framework(master_lattice=JFEL_DIR, directory=str(OUT / "periodic"), clean=True, verbose=False)
periodic.loadSettings(str(periodic_def))
periodic.track()
ring = periodic["Periodic"].ring_parameters()
target = {k: float(ring[k]) for k in KEYS}

# Detune, set the targets, then track: the match runs first
for q, k1l in zip(QUADS, stored):
    framework.modifyElement(q, "k1l", 0.5 * k1l)
framework["Match"].file_block["match"]["targets"] = {ENTRANCE: target}
framework.track(files=["Match", "Undulator"], check_lattice=False)
matched = [framework.getElement(q).k1l for q in QUADS]

rows = {"periodic (SIMBA)": target, "beam at entrance": twiss(ENTRANCE), "beam at exit": twiss(EXIT)}
rows.update({f"README {k}": v for k, v in README.items()})
print(f"\n{'':18s}" + "".join(f"{k:>9s}" for k in KEYS))
for label, row in rows.items():
    print(f"{label:18s}" + "".join(f"{row[k]:9.4f}" for k in KEYS))
print(f"\n{'S13 k1l [1/m]':18s}" + "".join(f"{q[-7:]:>9s}" for q in QUADS))
for label, values in (("stored", stored), ("start (x0.5)", [0.5 * k for k in stored]), ("matched", matched)):
    print(f"{label:18s}" + "".join(f"{v:9.4f}" for v in values))

# --- Checks ----------------------------------------------------------------
problems = []
for label, got, want, beta_tol, alpha_tol in (
    ("entrance vs target", rows["beam at entrance"], target, 0.02, 0.02),
    ("exit vs target", rows["beam at exit"], target, 0.02, 0.02),
    ("periodic vs README", target, README["periodic"], 1e-3, 0.002),
    ("entrance vs README", rows["beam at entrance"], README["entrance"], 1e-3, 0.002),
    ("exit vs README", rows["beam at exit"], README["exit"], 1e-3, 0.002),
):
    beta, alpha = differences(got, want)
    print(f"{label:20s} beta {beta:.1e} (limit {beta_tol}), alpha {alpha:.4f} (limit {alpha_tol})")
    if beta > beta_tol or alpha > alpha_tol:
        problems.append(label)
k_diff = max(abs(m / s - 1) for m, s in zip(matched, stored))
print(f"{'matched vs stored':20s} k1l {k_diff:.1e} (limit 1e-3)")
if k_diff > 1e-3:
    problems.append("matched k1l vs stored")
assert not problems, f"failed: {problems}"
print("OK: from half strength, the match block brings the beam onto the undulator's periodic Twiss")

## 24 A two-parameter scan in parallel

Chicane angle × L4H phase, 50 runs over a `ProcessPoolExecutor`. Each process builds one `Framework`, gives each run its own directory with `setSubDirectory`, and uses one thread (Ocelot's numpy threads would otherwise fight over the cores). The results go into a CSV. Straight-line fits to the two arms of each row of $\sigma_t$ find the phase of full compression: the README's "compression valley", rising steadily with the angle.

In [ ]:
__file__ = "24_parameter_scan_parallel.py"  # python and %run set this; a notebook cell does not
"""
24 - Scan two parameters in parallel
====================================

Shows: a grid scan run over several processes, one directory per run, with
the results gathered into a CSV. It scans JFEL's chicane angle against the
L4H cavity phase and maps the bunch length after the chicane: the
"compression valley" of the JFEL README.

Run:    python 04_scans_optimisation/24_parameter_scan_parallel.py
Tier:   A
Needs:  SIMBA, LAURA, laura-lattices (JFEL) and Ocelot.
Output: output/24_parameter_scan_parallel/: one directory per run
        (angle_phase/), scan.csv and valley.png.

Units: angle in rad; phase in degrees (SIMBA's convention); sigma_t in fs.

How it runs in parallel:
- concurrent.futures.ProcessPoolExecutor, one process per core. Each
  process builds one Framework when it starts (the pool's initializer),
  and every run then only changes settings, so the lattice is loaded once
  per process, not per run.
- framework.setSubDirectory(path) before each track() gives each run its
  own directory, so runs don't overwrite each other's files.
- Each process uses one thread (OMP_NUM_THREADS=1 and friends, set before
  numpy is imported). Ocelot is multithreaded through numpy, and 8
  processes x 8 threads on 8 cores runs twice as slowly as 8 x 1.
- sample_interval = 4 tracks every 4th particle, with the same total
  charge: 4x faster, and sigma_t changes by ~1 fs.

The line is JFEL's Lattices/jfel_compression.def, CSR off, wakes on (as
22). Each run calls set_angle on the chicane group (which also sets the
dipole lengths; see ISSUES.md #27) and sets the L4H phase.

Check: the valley. Along each angle's row, sigma_t falls linearly to a
rounded floor at full compression and rises linearly after it. Straight
lines fitted to the two arms (points above twice the floor) meet at the
phase of full compression, phi_c. It must rise steadily with the angle: a
bigger angle gives a bigger R56, so full compression needs a different
chirp from L4H (measured ~0.9 deg per mrad). And it must be within 5 deg
(one grid step) of the README's "L4H phase at full compression", read off
the same grid by its minimum. Measured 2.1-2.9 deg, every one above the
README's grid value.
"""

import os

for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_var] = "1"

import csv  # noqa: E402
import itertools  # noqa: E402
import time  # noqa: E402
from concurrent.futures import ProcessPoolExecutor  # noqa: E402
from pathlib import Path  # noqa: E402

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
L4H, END = "JFEL-L4H-LIN-CAV-01", "JFEL-S06-SIM-MARK-01"
ANGLES = [0.120, 0.125, 0.130, 0.135, 0.140]
PHASES = np.arange(-200.0, -154.0, 5.0)
README_VALLEY = {0.120: -190, 0.125: -185, 0.130: -180, 0.135: -175, 0.140: -170}
framework = None  # one per worker process


def start_worker():
    global framework
    import simba.Framework as fw
    from laura_lattices import JFEL

    framework = fw.Framework(master_lattice=os.path.dirname(JFEL.__file__),
                             directory=str(OUT / f"worker_{os.getpid()}"), clean=True, verbose=False)
    framework.loadSettings("Lattices/jfel_compression.def")
    framework.change_Lattice_Code("Compressor", "ocelot")
    framework["Compressor"].csr_enable = False
    framework["Compressor"].sample_interval = 4


def run(point):
    """One scan point, in a worker: set, track into its own directory, return sigma_t [fs]."""
    import simba.Modules.Beams as rbf

    angle, phase = point
    directory = OUT / f"{angle:.3f}_{phase:+.0f}"
    framework.setSubDirectory(str(directory))
    framework["bunch_compressor"].set_angle(angle)
    framework.modifyElement(L4H, "phase", float(phase))
    framework.track(files=["Compressor"], check_lattice=False)
    beam = rbf.load_file(str(directory / f"{END}.openpmd.hdf5"))
    sigma_t = np.sqrt(np.cov(beam.t.val, aweights=np.abs(beam.Particles.charge.val)))
    return angle, float(phase), sigma_t * 1e15, str(directory.relative_to(OUT))


def full_compression_phase(sigma_t):
    """Where straight lines through the two arms of a row meet."""
    low = np.argmin(sigma_t)
    arm = sigma_t > 2 * sigma_t[low]
    left, right = arm & (np.arange(len(PHASES)) < low), arm & (np.arange(len(PHASES)) > low)
    assert left.sum() >= 2 and right.sum() >= 2, f"not enough points on the arms: {sigma_t.round()}"
    (m1, c1), (m2, c2) = np.polyfit(PHASES[left], sigma_t[left], 1), np.polyfit(PHASES[right], sigma_t[right], 1)
    return (c2 - c1) / (m1 - m2)


if __name__ == "__main__":
    OUT.mkdir(parents=True, exist_ok=True)
    points = list(itertools.product(ANGLES, PHASES))
    workers = os.cpu_count()
    t0 = time.time()
    with ProcessPoolExecutor(workers, initializer=start_worker) as pool:
        results = list(pool.map(run, points))
    print(f"{len(points)} runs on {workers} processes in {time.time() - t0:.0f} s")

    with open(OUT / "scan.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["angle_rad", "l4h_phase_deg", "sigma_t_fs", "directory"])
        writer.writerows(results)
    grid = np.array([r[2] for r in results]).reshape(len(ANGLES), len(PHASES))

    print(f"\nsigma_t [fs]; phases {PHASES[0]:.0f} to {PHASES[-1]:.0f} deg in steps of 5")
    valley = {}
    for angle, row in zip(ANGLES, grid):
        valley[angle] = full_compression_phase(row)
        print(f"  {angle:.3f} rad: " + " ".join(f"{s:4.0f}" for s in row)
              + f"   phi_c {valley[angle]:7.2f} (README {README_VALLEY[angle]})")

    plt.pcolormesh(PHASES, ANGLES, grid, shading="nearest", cmap="viridis")
    plt.colorbar(label="sigma_t after the chicane [fs]")
    plt.plot(list(valley.values()), ANGLES, "w.-", label="full compression (arm fits)")
    plt.plot(list(README_VALLEY.values()), ANGLES, "rx", label="JFEL README")
    plt.xlabel("L4H phase [deg]")
    plt.ylabel("chicane angle [rad]")
    plt.legend()
    plt.savefig(OUT / "valley.png", dpi=120, bbox_inches="tight")
    print(f"\nCSV: {OUT / 'scan.csv'}\nPlot: {OUT / 'valley.png'}")

    # --- Check -------------------------------------------------------------
    phi = np.array([valley[a] for a in ANGLES])
    assert np.all(np.diff(phi) > 0), f"full-compression phase does not rise with angle: {phi.round(2)}"
    worst = max(abs(valley[a] - README_VALLEY[a]) for a in ANGLES)
    assert worst <= 5.0, f"valley {worst:.1f} deg from the README's"
    print(f"OK: the valley rises with the angle and is within {worst:.1f} deg of the README's")

In [ ]:
display(Image(filename="../output/24_parameter_scan_parallel/valley.png"))